# Fabric Capacity Management

This notebook helps manage Microsoft Fabric capacities.

## What this notebook does:
- Lists all available capacities
- Shows capacity utilization and details
- Assigns workspaces to capacities
- Exports capacity information for reporting

In [ ]:
# Import required modules
import sys
sys.path.append('..')

from modules.fabric_auth import FabricAuth, load_credentials
from modules.fabric_client import FabricClient, capacities_to_dataframe, workspaces_to_dataframe
from modules.utils import export_to_excel
import pandas as pd

In [ ]:
# Load credentials and authenticate
credentials = load_credentials('../config/credentials.json')
auth = FabricAuth(
    tenant_id=credentials['tenant_id'],
    client_id=credentials['client_id'],
    client_secret=credentials['client_secret']
)

token = auth.get_access_token()
if token:
    print("✓ Authentication successful")
    client = FabricClient(token)
else:
    print("✗ Authentication failed")

In [ ]:
# Get all capacities
print("Fetching capacities...")
capacities = client.get_capacities()

if capacities:
    print(f"✓ Found {len(capacities)} capacities")
    capacities_df = capacities_to_dataframe(capacities)
    display(capacities_df)
else:
    print("✗ No capacities found or error occurred")

In [ ]:
# Get all workspaces to see capacity assignments
print("Fetching workspaces...")
workspaces = client.get_workspaces()

if workspaces:
    print(f"✓ Found {len(workspaces)} workspaces")
    workspaces_df = workspaces_to_dataframe(workspaces)
    
    # Filter to show relevant columns
    columns_to_show = ['id', 'name', 'capacityId', 'type', 'state']
    available_columns = [col for col in columns_to_show if col in workspaces_df.columns]
    
    if available_columns:
        display(workspaces_df[available_columns])
    else:
        display(workspaces_df)
else:
    print("✗ No workspaces found or error occurred")

In [ ]:
# Analyze capacity assignments
if workspaces and 'capacityId' in workspaces_df.columns:
    print("\n=== Capacity Assignment Summary ===")
    
    # Count workspaces per capacity
    capacity_counts = workspaces_df['capacityId'].value_counts()
    print(f"\nWorkspaces per capacity:")
    print(capacity_counts)
    
    # Workspaces not assigned to any capacity
    unassigned = workspaces_df[workspaces_df['capacityId'].isna()]
    print(f"\nWorkspaces not assigned to capacity: {len(unassigned)}")
    if len(unassigned) > 0:
        print("Unassigned workspaces:")
        display(unassigned[['id', 'name']])

In [ ]:
# Assign workspace to capacity (example)
# Uncomment and modify to assign a workspace to a capacity

# workspace_id = "YOUR_WORKSPACE_ID"
# capacity_id = "YOUR_CAPACITY_ID"

# result = client.assign_workspace_to_capacity(workspace_id, capacity_id)
# if result:
#     print(f"✓ Workspace assigned to capacity successfully")
# else:
#     print(f"✗ Failed to assign workspace to capacity")

In [ ]:
# Export capacity and workspace data
if capacities and workspaces:
    from datetime import datetime
    
    # Create a combined report
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_filename = f"fabric_capacity_report_{timestamp}.xlsx"
    
    # Export to Excel with multiple sheets
    with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
        capacities_df.to_excel(writer, sheet_name='Capacities', index=False)
        workspaces_df.to_excel(writer, sheet_name='Workspaces', index=False)
    
    print(f"\n✓ Capacity report exported to {output_filename}")